# 06. RBAC & ABAC

## Background

Role-Based Access Control (RBAC) is the dominant authorization model in enterprise systems: assign users to roles, grant permissions to roles, check role membership at runtime. It is simple, auditable, and well-understood. Its failure mode is **role explosion**: as systems grow, the number of roles multiplies to handle every combination of department, project, and sensitivity level until the role graph is unmanageable.

Attribute-Based Access Control (ABAC) generalizes RBAC by making access decisions based on arbitrary attributes of the subject (user), resource, and environment. This enables fine-grained policies that RBAC cannot express without combinatorial role proliferation. The Open Policy Agent (OPA) is the community standard for ABAC in cloud-native systems.

## What You'll Learn

- RBAC implementation: roles, permissions, role hierarchy
- ABAC policy language: subject + resource + environment attributes
- OPA Rego policy language basics (simulated in Python)
- Building an ML API authorization layer with ABAC policies
- Role explosion diagnosis and mitigation

## Why This Matters

OPA is used by Kubernetes, Envoy, and every major cloud-native authorization system. The Rego policy language is the standard for expressing ABAC policies as code. For LLM applications, ABAC enables per-user tool scoping, data classification-based access, and time/geo-conditional permissions — things RBAC cannot express without hundreds of roles.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional
import json, time
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

plt.style.use('seaborn-v0_8-whitegrid')
print('RBAC/ABAC demo ready')


## 1. RBAC Implementation

In [ ]:
@dataclass
class RBACEngine:
    _role_permissions: dict = field(default_factory=dict)
    _role_hierarchy:   dict = field(default_factory=dict)
    _user_roles:       dict = field(default_factory=dict)

    def add_role(self, role, permissions, parent=None):
        self._role_permissions[role] = set(permissions)
        if parent:
            self._role_hierarchy[role] = parent

    def assign_role(self, user_id, role):
        self._user_roles.setdefault(user_id, set()).add(role)

    def _effective_permissions(self, role, visited=None):
        if visited is None: visited = set()
        if role in visited: return set()
        visited.add(role)
        perms = set(self._role_permissions.get(role, []))
        parent = self._role_hierarchy.get(role)
        if parent:
            perms |= self._effective_permissions(parent, visited)
        return perms

    def can(self, user_id, permission):
        for role in self._user_roles.get(user_id, []):
            if permission in self._effective_permissions(role):
                return True
        return False

    def user_permissions(self, user_id):
        perms = set()
        for role in self._user_roles.get(user_id, []):
            perms |= self._effective_permissions(role)
        return perms


rbac = RBACEngine()
rbac.add_role('viewer',     ['read:data', 'read:reports'])
rbac.add_role('analyst',    ['run:models', 'read:features'],  parent='viewer')
rbac.add_role('ml_engineer',['deploy:models','write:data'],    parent='analyst')
rbac.add_role('admin',      ['manage:users','delete:data'],    parent='ml_engineer')

rbac.assign_role('alice', 'analyst')
rbac.assign_role('bob',   'viewer')
rbac.assign_role('carol', 'admin')

for user in ['alice','bob','carol']:
    perms = sorted(rbac.user_permissions(user))
    print(f'{user}: {perms}')

print()
tests = [('alice','deploy:models'),('alice','read:data'),('bob','run:models'),('carol','delete:data')]
for user, perm in tests:
    ok = rbac.can(user, perm)
    print(f'  {user} can {perm}: {ok}')


## 2. ABAC Policy Engine (OPA/Rego-style in Python)

ABAC policies evaluate subject attributes (user), resource attributes (data classification), and environment attributes (time, network) together. This eliminates role explosion: instead of 'admin_on_pii_in_us_during_business_hours', you write a policy rule.

In [ ]:
@dataclass
class Subject:
    user_id:    str
    department: str
    clearance:  int     # 1=public, 2=internal, 3=confidential, 4=secret
    mfa:        bool
    location:   str     # 'office','vpn','remote'

@dataclass
class Resource:
    resource_id:     str
    classification:  int   # matches clearance levels
    owner_dept:      str
    action:          str   # 'read','write','delete','run'

@dataclass
class Environment:
    hour:    int
    day:     str  # 'weekday','weekend'
    network: str  # 'internal','external'


class ABACEngine:
    def __init__(self):
        self._policies = []

    def add_policy(self, name, rule_fn):
        self._policies.append((name, rule_fn))

    def evaluate(self, subject, resource, env):
        results = []
        for name, rule in self._policies:
            decision, reason = rule(subject, resource, env)
            results.append({'policy': name, 'decision': decision, 'reason': reason})
        # Default deny; allow only if at least one ALLOW and no DENY
        allows = [r for r in results if r['decision'] == 'ALLOW']
        denies  = [r for r in results if r['decision'] == 'DENY']
        if denies:  return False, denies[0]['reason'], results
        if allows:  return True,  allows[0]['reason'],  results
        return False, 'No matching allow policy', results


engine = ABACEngine()

# Policy 1: clearance must meet or exceed classification
engine.add_policy('clearance_check',
    lambda s,r,e: ('ALLOW','Clearance sufficient') if s.clearance >= r.classification
                  else ('DENY', f'Clearance {s.clearance} < classification {r.classification}'))

# Policy 2: MFA required for confidential+
engine.add_policy('mfa_for_confidential',
    lambda s,r,e: ('ALLOW','MFA ok') if r.classification < 3 or s.mfa
                  else ('DENY', 'MFA required for confidential data'))

# Policy 3: delete requires admin clearance regardless
engine.add_policy('delete_requires_clearance4',
    lambda s,r,e: ('ALLOW','Delete allowed') if r.action != 'delete' or s.clearance >= 4
                  else ('DENY', 'Delete requires clearance 4'))

# Policy 4: business hours for writes from remote
engine.add_policy('remote_writes_business_hours',
    lambda s,r,e: ('ALLOW','Business hours ok') if not (s.location=='remote' and r.action=='write')
                  or (e.day=='weekday' and 8<=e.hour<=18)
                  else ('DENY', 'Remote writes only during business hours'))

scenarios = [
    (Subject('alice','ml',3,True,'office'),  Resource('pii_data','pii',3,'ml','read'),   Environment(10,'weekday','internal')),
    (Subject('bob','ml',2,False,'remote'),   Resource('model_v2','models',3,'ml','read'), Environment(14,'weekday','external')),
    (Subject('carol','ml',4,True,'remote'),  Resource('raw_data','ml',2,'ml','write'),    Environment(22,'weekend','external')),
    (Subject('dave','ops',3,True,'office'),  Resource('secret_db','db',4,'ops','delete'), Environment(11,'weekday','internal')),
]

for subj, res, env in scenarios:
    ok, reason, _ = engine.evaluate(subj, res, env)
    print(f'[{"ALLOW" if ok else "DENY "}] {subj.user_id} {res.action} {res.resource_id}  — {reason}')


## 3. OPA Rego Policy Patterns

Real deployments use OPA with Rego. Below are the Python-equivalent patterns translated to pseudo-Rego for reference.

In [ ]:
rego_examples = '''
# Rego equivalent of clearance_check policy
package ml_api

default allow = false

# Allow read if clearance >= classification
allow {
    input.action == "read"
    input.subject.clearance >= input.resource.classification
    input.subject.mfa == true
}

# Allow write only business hours for remote users
allow {
    input.action == "write"
    input.subject.clearance >= input.resource.classification
    not is_remote_after_hours
}

is_remote_after_hours {
    input.subject.location == "remote"
    not business_hours
}

business_hours {
    input.env.day == "weekday"
    input.env.hour >= 8
    input.env.hour <= 18
}
'''
print('Rego policy for ML API authorization:')
print(rego_examples)
print('Deploy: opa run --server policy.rego')
print('Query:  curl -X POST http://localhost:8181/v1/data/ml_api/allow -d @input.json')
